# Code_5 — Real LLM-Generated Tool Calls and Cross-Model Evaluation

**GPU: yes** (long; generation is resumable — re-run the cell after a disconnect and it continues)

Real calls are generated by several instruction-tuned models for the 21 ToolAlpaca evaluation tools
(214 queries with gold calls; queries whose gold answer is a clarification request are reported but not scored).
The registry contains the schemas of all these tools, so this is **not** the "unknown function" artifact
of the previous version (#9, #13). Conditions:

| Condition | What is measured |
|---|---|
| raw model output | how often each model is right on its own |
| + schema-only pipeline | detection / classification / correction of real errors |
| constrained decoding | function name chosen from the tool's functions, keys forced by the schema, values generated (#8) |
| out-of-registry control | the same pipeline with these tools REMOVED from the registry (reproduces the old artifact) |

Tools are split into **known** (some of their gold calls were used to calibrate type trust) and
**unseen** (never used for any tuning). Automatic labels come from comparing each call with the gold
call (`reference_labels`); a subset is labelled by two humans in Code_6, which replaces these labels.

Outputs: `runs/<run_id>/real_results.json`, `runs/<run_id>/real_records.json`; generations cached in
`cache/real_generations/` (never regenerated once present).

In [ ]:
import os, sys, json, importlib, subprocess, time, copy, random, re
from collections import Counter, defaultdict

IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/hallucination_project'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers', 'rapidfuzz', 'accelerate'],
                   check=True)
else:
    DRIVE_ROOT = os.environ.get('TOOLHALL_ROOT', os.path.abspath('hallucination_project'))
CODE_DIR = os.path.join(DRIVE_ROOT, 'code')
sys.path.insert(0, CODE_DIR)
import toolhall_core as th
importlib.reload(th)
import numpy as np

NOTEBOOK = 'Code_5'
CFG = th.load_config(os.path.join(CODE_DIR, 'config.json'))
CFG['drive_root'] = DRIVE_ROOT
P = th.Paths(CFG)
SEED = CFG['seed']
th.set_seed(SEED)
NB, ALPHA = CFG['bootstrap']['n'], CFG['bootstrap']['alpha']
MODELS = CFG['models']['generation_models']
RUN_CONSTRAINED = True     # constrained decoding is slower (one pass per candidate function and per parameter)
RUN_JUDGE = False          # optional automatic intent judge (reported as automatic, never as human)
QUICK_TEST = None          # e.g. 10 = trial run on 10 queries; None = full run
GEN_DIR = P.cache_file('real_generations')
os.makedirs(GEN_DIR, exist_ok=True)

raw_registry = th.load_json(P.data_file('registry.json'), CFG, kind='data')
real = th.load_json(P.data_file('real_eval.json'), CFG, kind='data')
single = th.load_json(P.data_file('benchmark_single.json'), CFG, kind='data')
splits = th.load_json(P.data_file('splits.json'), CFG, kind='data')
core = th.load_json(P.run_file('core_results.json'), CFG)
REG = th.Registry(raw_registry, CFG['validator']['unknown_required_policy'])
EVAL = [r for r in real if r['role'] == 'eval']
CALIB = [r for r in real if r['role'] == 'calibration']
if QUICK_TEST:
    EVAL = EVAL[:QUICK_TEST]
print(f"models: {MODELS}")
print(f"real queries: {len(real)} | eval {len(EVAL)} | calibration {len(CALIB)} | "
      f"clarification (not scored) {sum(r['role'] == 'excluded_clarification' for r in real)}")
print('eval by tool group:', dict(Counter(r['tool_group'] for r in EVAL)))

## Prompt

Every model gets the same prompt: the tool's functions with their full schemas (name, description,
parameters with type / required / enum), and the request. It must answer with exactly one JSON object.

In [ ]:
SYSTEM_PROMPT = ('You are a function-calling assistant. Choose ONE of the available functions and reply with '
                 'exactly one JSON object of the form {"function": "<name>", "parameters": {...}} and nothing else.')


def messages_for(r):
    tools = [th.schema_for_prompt(REG, f) for f in r['tool_functions'] if REG.resolve(f)]
    return [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f"Available functions:\n{json.dumps(tools, ensure_ascii=False)[:6000]}\n\n"
                                        f"Request: {r['query']}"}]


print(messages_for(EVAL[0])[1]['content'][:1500])

## Generate (resumable, cached per model)

In [ ]:
def gen_path(model, mode):
    return os.path.join(GEN_DIR, f"{model.replace('/', '_')}__{mode}.jsonl")


import torch
for model_name in MODELS:
    todo_free = [r for r in EVAL if r['id'] not in {x['id'] for x in th.read_jsonl(gen_path(model_name, 'free'))}]
    todo_con = [r for r in EVAL if r['id'] not in {x['id'] for x in th.read_jsonl(gen_path(model_name, 'constrained'))}] \
        if RUN_CONSTRAINED else []
    if not todo_free and not todo_con:
        print(f"{model_name}: cached")
        continue
    print(f"{model_name}: {len(todo_free)} free + {len(todo_con)} constrained generations to do")
    try:
        tok, model, device = th.load_causal_lm(model_name)
    except Exception as e:
        print(f"   could not load {model_name}: {type(e).__name__}: {e} -- skipped")
        continue
    for start in range(0, len(todo_free), 16):
        chunk = todo_free[start:start + 16]
        prompts = [th.chat_text(tok, messages_for(r)) for r in chunk]
        texts, secs = th.generate_batch(tok, model, device, prompts, CFG['models']['max_new_tokens'], batch_size=8)
        for r, t, s in zip(chunk, texts, secs):
            th.append_jsonl(gen_path(model_name, 'free'), {'id': r['id'], 'output': t, 'seconds': s})
        print(f"   free {min(start + 16, len(todo_free))}/{len(todo_free)}")
    for k, r in enumerate(todo_con):
        fns = [f for f in r['tool_functions'] if REG.resolve(f)]
        t0 = time.perf_counter()
        try:
            call = th.constrained_generate(tok, model, device, th.chat_text(tok, messages_for(r)), fns, REG)
            err = None
        except Exception as e:
            call, err = None, f"{type(e).__name__}: {e}"
        th.append_jsonl(gen_path(model_name, 'constrained'),
                        {'id': r['id'], 'output': json.dumps(call) if call else '', 'seconds': time.perf_counter() - t0,
                         'error': err})
        if (k + 1) % 25 == 0:
            print(f"   constrained {k + 1}/{len(todo_con)}")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Pipeline configuration for real data

The function-disjoint tuned configuration from Code_3 (the conservative choice for unseen tools).
Type trust is fitted on the synthetic tune clean calls **plus** the gold calls reserved for calibration
(known tools only). Unseen tools therefore get no tool-specific calibration.

In [ ]:
t = core['tuned']['function_disjoint']
tune_clean = [th.normalize_tool_call(single[i]['generated_tool_call'])
              for i in splits['function_disjoint']['tune'] if not single[i]['is_hallucinated']]
cal_calls = [r['gt_call'] for r in CALIB]


def build_pipe(registry):
    tt = th.TypeTrust(mode='adaptive', tau=t['tau'], min_obs=CFG['type_trust']['min_obs'],
                      prior_agree=CFG['type_trust']['prior_agree'],
                      prior_disagree=CFG['type_trust']['prior_disagree']).fit(tune_clean + cal_calls, registry)
    m = th.FunctionMatcher(registry, model_name=CFG['matcher']['model_name'], cache_dir=P.cache, verbose=False,
                           use_semantic=t['w_semantic'] > 0, use_edit=t['w_semantic'] < 1.0,
                           w_semantic=t['w_semantic'], min_score=t['min_score'])
    return th.Pipeline(registry, CFG, type_trust=tt, matcher=m,
                       corrector_overrides={'max_iterations': t['max_iterations']})


PIPE = build_pipe(REG)
eval_tool_functions = sorted({f for r in real for f in r['tool_functions']})
REG_OUT = th.Registry(th.registry_without(raw_registry, eval_tool_functions), CFG['validator']['unknown_required_policy'])
PIPE_OUT = build_pipe(REG_OUT)
print(f"pipeline ready | out-of-registry control removes {len(eval_tool_functions)} functions")

## Score every model and condition

Automatic labels: `reference_labels(generated, gold)` — multi-label, with WRONG_FUNCTION split into
*not in registry* vs *wrong existing function* (a real model can pick a valid but wrong function,
which no schema can detect). Exact match is strict (envelope + values); a **lenient** match that ignores
case/whitespace in strings and int/float/str spelling of numbers is also reported as an intent proxy.

In [ ]:
def lenient_equal(a, b):
    na, nb = th.normalize_tool_call(a), th.normalize_tool_call(b)
    if na is None or nb is None or na['function'] != nb['function']:
        return False
    pa = {k: v for k, v in na['parameters'].items() if v not in ('', None)}
    pb = {k: v for k, v in nb['parameters'].items() if v not in ('', None)}
    if set(pa) != set(pb):
        return False
    for k in pa:
        x, y = pa[k], pb[k]
        try:
            if float(x) == float(y):
                continue
        except (TypeError, ValueError):
            pass
        if str(x).strip().lower() != str(y).strip().lower():
            return False
    return True


GOLD = {r['id']: r for r in EVAL}
RECORDS = []
for model_name in MODELS:
    for mode in (['free', 'constrained'] if RUN_CONSTRAINED else ['free']):
        gens = {x['id']: x for x in th.read_jsonl(gen_path(model_name, mode))}
        if not gens:
            continue
        for rid, r in GOLD.items():
            g = gens.get(rid)
            if g is None:
                continue
            out = g['output']
            parsed = th.parse_raw_call(out)
            labels, det = th.reference_labels(out, r['gt_call'], REG)
            ex = {'id': rid, 'generated_tool_call': out if parsed is None else parsed,
                  'ground_truth_tool_call': r['gt_call'], 'query': r['query'], 'error_types': labels}
            for cond, pipe in (('pipeline', PIPE), ('out_of_registry_control', PIPE_OUT)):
                if cond == 'out_of_registry_control' and mode != 'free':
                    continue
                rec = pipe.run_one(ex)
                rec.update({'model': model_name, 'mode': mode, 'condition': cond, 'tool_group': r['tool_group'],
                            'tool_name': r['tool_name'], 'auto_labels': labels, 'auto_label_detail': det,
                            'parseable': parsed is not None, 'raw_output': out[:1500],
                            'gen_seconds': g.get('seconds'),
                            'raw_exact': th.exact_strict(parsed, r['gt_call']) if parsed else False,
                            'raw_lenient': lenient_equal(parsed, r['gt_call']) if parsed else False,
                            'final_lenient': lenient_equal(rec['final_call'], r['gt_call']) if rec['final_call'] else False,
                            'raw_schema_valid': parsed is not None and len(PIPE.schema_checker.validate(parsed)) == 0})
                RECORDS.append(rec)
print(f"{len(RECORDS)} scored records")

## Results

In [ ]:
def table(recs):
    if not recs:
        return None
    y = [r['is_hallucinated'] for r in recs]
    f = [r['flagged'] for r in recs]
    b = th.binary_metrics(y, f)
    ml = th.multilabel_metrics([set(r['true_types']) for r in recs if r['is_hallucinated']],
                               [set(r['pred_types']) for r in recs if r['is_hallucinated']])
    mean = lambda k: float(np.mean([bool(r[k]) for r in recs]))
    return {'n': len(recs), 'parse_rate': mean('parseable'), 'raw_error_rate': float(np.mean(y)),
            'raw_schema_valid': mean('raw_schema_valid'), 'raw_exact': mean('raw_exact'), 'raw_lenient': mean('raw_lenient'),
            'detection': b, 'error_type_macro_f1': ml['macro_f1'], 'error_type_set_accuracy': ml['exact_set_accuracy'],
            'final_schema_valid': mean('final_schema_valid'), 'final_exact': mean('final_exact'),
            'final_lenient': mean('final_lenient'),
            'final_exact_ci': th.bootstrap_ci(lambda x: float(np.mean(x)), [int(r['final_exact']) for r in recs],
                                              n=NB, alpha=ALPHA, seed=SEED),
            'mcnemar_raw_vs_final_exact': th.mcnemar_exact([int(r['final_exact']) for r in recs],
                                                           [int(r['raw_exact']) for r in recs]),
            'auto_label_distribution': dict(Counter(t for r in recs for t in r['auto_labels'])),
            'wrong_function_subtypes': dict(Counter(r['auto_label_detail'].get('WRONG_FUNCTION') for r in recs
                                                    if 'WRONG_FUNCTION' in r['auto_labels'])),
            'mean_gen_seconds': float(np.mean([r['gen_seconds'] or 0 for r in recs]))}


REAL = {}
th.print_header("REAL LLM CALLS — per model (pipeline condition)")
print(f"{'model':<38}{'mode':<12}{'group':<8}{'n':>4}{'parse':>7}{'errRate':>8}{'rawEx':>7}{'detF1':>7}"
      f"{'FPR':>6}{'typeF1':>7}{'finSV':>7}{'finEx':>7}{'finLen':>7}")
for model_name in MODELS:
    for mode in ['free', 'constrained']:
        for grp in ['all', 'known', 'unseen']:
            recs = [r for r in RECORDS if r['model'] == model_name and r['mode'] == mode and r['condition'] == 'pipeline'
                    and (grp == 'all' or r['tool_group'] == grp)]
            tb = table(recs)
            if tb is None:
                continue
            REAL[f"{model_name}|{mode}|{grp}"] = tb
            print(f"{model_name[-38:]:<38}{mode:<12}{grp:<8}{tb['n']:>4}{tb['parse_rate']:>7.2f}{tb['raw_error_rate']:>8.2f}"
                  f"{tb['raw_exact']:>7.3f}{tb['detection']['f1']:>7.3f}{tb['detection']['fpr']:>6.2f}"
                  f"{tb['error_type_macro_f1']:>7.3f}{tb['final_schema_valid']:>7.3f}{tb['final_exact']:>7.3f}"
                  f"{tb['final_lenient']:>7.3f}")

th.print_header("OUT-OF-REGISTRY CONTROL (the previous experiment's setting) vs registry WITH the tool schemas")
print(f"{'model':<38}{'condition':<26}{'flag rate':>10}{'final exact':>12}")
CONTROL = {}
for model_name in MODELS:
    for cond in ['pipeline', 'out_of_registry_control']:
        recs = [r for r in RECORDS if r['model'] == model_name and r['mode'] == 'free' and r['condition'] == cond]
        if recs:
            fr = float(np.mean([r['flagged'] for r in recs]))
            fe = float(np.mean([r['final_exact'] for r in recs]))
            CONTROL[f"{model_name}|{cond}"] = {'flag_rate': fr, 'final_exact': fe,
                                               'flag_rate_on_correct_raw_calls': float(np.mean(
                                                   [r['flagged'] for r in recs if not r['is_hallucinated']]))
                                               if any(not r['is_hallucinated'] for r in recs) else None}
            print(f"{model_name[-38:]:<38}{cond:<26}{fr:>10.3f}{fe:>12.3f}")

## Optional: automatic intent judge (reported as automatic, never as human)

In [ ]:
JUDGE = {}
if RUN_JUDGE:
    tok, model, device = th.load_causal_lm(CFG['models']['judge_model'])
    jpath = P.cache_file(f"judge_{CFG['models']['judge_model'].replace('/', '_')}.jsonl")
    done = {(x['id'], x['model'], x['mode']): x for x in th.read_jsonl(jpath)}
    todo = [r for r in RECORDS if r['condition'] == 'pipeline' and (r['id'], r['model'], r['mode']) not in done]
    for start in range(0, len(todo), 16):
        chunk = todo[start:start + 16]
        prompts = [th.chat_text(tok, [{'role': 'user', 'content':
                   f"Request: {GOLD[r['id']]['query']}\nTool call: {json.dumps(r['final_call'], default=str)[:800]}\n"
                   f"Does this tool call do what the request asks? Answer YES or NO."}]) for r in chunk]
        texts, _ = th.generate_batch(tok, model, device, prompts, 4, 8)
        for r, txt in zip(chunk, texts):
            rec = {'id': r['id'], 'model': r['model'], 'mode': r['mode'], 'answer': txt}
            th.append_jsonl(jpath, rec)
            done[(r['id'], r['model'], r['mode'])] = rec
    for r in RECORDS:
        if r['condition'] == 'pipeline':
            a = done.get((r['id'], r['model'], r['mode']), {}).get('answer', '')
            r['judge_intent_yes'] = a.strip().upper().startswith('YES')
    for key in REAL:
        m, mode, grp = key.split('|')
        rs = [r for r in RECORDS if r['model'] == m and r['mode'] == mode and r['condition'] == 'pipeline'
              and (grp == 'all' or r['tool_group'] == grp)]
        JUDGE[key] = float(np.mean([r.get('judge_intent_yes', False) for r in rs]))
    print('judge intent-yes rate (automatic):', {k: round(v, 3) for k, v in JUDGE.items()})

## Save

In [ ]:
gold_viol = Counter(v['type'] for r in EVAL for v in PIPE.schema_checker.validate(r['gt_call']))
th.save_json({'per_model': REAL, 'out_of_registry_control': CONTROL, 'judge_automatic': JUDGE,
              'gold_call_schema_violations': dict(gold_viol), 'n_eval': len(EVAL),
              'n_clarification_excluded': sum(r['role'] == 'excluded_clarification' for r in real),
              'models': MODELS, 'prompt_system': SYSTEM_PROMPT, 'tuned_config_used': t,
              'hardware': th.hardware_info(), 'packages': th.package_versions()},
             P.run_file('real_results.json'), CFG, NOTEBOOK)
th.save_json(RECORDS, P.run_file('real_records.json'), CFG, NOTEBOOK)
print('gold calls that themselves violate the schema:', dict(gold_viol))
print('saved ->', P.run_file('real_results.json'))
print('Code_5 complete.')